# 🚦 Traffic Sign Detection - YOLO26 Training

This notebook trains a YOLO26 model on the Zalo Traffic Sign dataset.
It handles:
1. Dataset preparation (COCO -> YOLO conversion)
2. Model training
3. Export and Visualization

In [1]:
# Install dependencies
%pip install ultralytics kagglehub==0.4.0
import ultralytics
ultralytics.checks()

Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.7/112.6 GB disk)


## 1. Dataset Setup

**Step 1:** Download your dataset (or ensure it is uploaded).
Set `dataset_path` to the folder containing your clean dataset (must have `annotations.json` and `images/` folder).

In [2]:
import os
import kagglehub
from getpass import getpass

# 1. Nhập thông tin (Nó sẽ hiện ô input phía trên hoặc dưới cell để bạn paste vào)
print("Nhập thông tin từ file kaggle.json của bạn:")

# Nhập Username (trong file json ghi là "username")
user = input("Username (ví dụ duongthanhduy): ")
os.environ['KAGGLE_USERNAME'] = user

# Nhập Key (trong file json ghi là "key")
# Dùng getpass để khi paste key vào nó sẽ ẩn đi (hiện dấu ***) cho bảo mật
key = getpass("Key (dãy ký tự dài): ")
os.environ['KAGGLE_KEY'] = key

# 2. Tải dataset
print(f"\nĐang thử tải dataset với tài khoản: {user} ...")
try:
    path = kagglehub.dataset_download("duongthanhduy/zaloaichallenge2020")
    print(f"✅ Tuyệt vời! Dataset đã tải xong tại: {path}")
except Exception as e:
    print(f"❌ Vẫn chưa được. Lỗi là: {e}")
    print("👉 Kiểm tra kỹ xem Key có bị copy thừa khoảng trắng không nhé.")

Nhập thông tin từ file kaggle.json của bạn:
Username (ví dụ duongthanhduy): duongthanhduy
Key (dãy ký tự dài): ··········

Đang thử tải dataset với tài khoản: duongthanhduy ...


100%|██████████| 9.21G/9.21G [07:12<00:00, 22.9MB/s]

Extracting files...


✅ Tuyệt vời! Dataset đã tải xong tại: /root/.cache/kagglehub/datasets/duongthanhduy/zaloaichallenge2020/versions/1


## 2. Convert Data to YOLO Format

This step converts the COCO-format `annotations.json` into YOLO textual labels and creates the directory structure required for training.

In [3]:
import json
import shutil
import os
import random
import yaml
from glob import glob

def auto_convert_zalo_to_yolo(root_path, output_path, split_ratio=0.8):
    print(f"🔍 Đang quét dataset tại: {root_path}")

    # 1. Tự động tìm file JSON (Labels)
    # Tìm tất cả file .json trong thư mục con
    json_files = [y for x in os.walk(root_path) for y in glob(os.path.join(x[0], '*.json'))]

    if not json_files:
        print("❌ Lỗi: Không tìm thấy file .json nào trong thư mục này!")
        return None

    # Lấy file json đầu tiên tìm được (Thường bộ Zalo chỉ có 1 file chính)
    json_file_path = json_files[0]
    print(f"   ✓ Đã tìm thấy file nhãn: {json_file_path}")

    # 2. Tự động tìm thư mục chứa ảnh (Images)
    # Tìm folder nào chứa nhiều file .png hoặc .jpg nhất
    image_dir = None
    max_images = 0

    for root, dirs, files in os.walk(root_path):
        count = sum(1 for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg')))
        if count > max_images:
            max_images = count
            image_dir = root

    if not image_dir:
        print("❌ Lỗi: Không tìm thấy thư mục nào chứa ảnh!")
        return None

    print(f"   ✓ Đã tìm thấy thư mục ảnh: {image_dir} ({max_images} ảnh)")

    # ---------------------------------------------------------
    # BẮT ĐẦU CONVERT (Dùng đúng path vừa tìm được)
    print("\n🚀 Bắt đầu chuyển đổi sang YOLO format...")

    with open(json_file_path, 'r') as f:
        coco_data = json.load(f)

    annotations = coco_data['annotations']
    images = {img['id']: img for img in coco_data['images']}
    categories = {cat['id']: cat for cat in coco_data['categories']}

    # Group annotations
    image_annotations = {}
    for ann in annotations:
        img_id = ann['image_id']
        if img_id not in image_annotations:
            image_annotations[img_id] = []
        image_annotations[img_id].append(ann)

    # Create Output Dirs
    for split in ['train', 'val']:
        os.makedirs(os.path.join(output_path, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_path, 'labels', split), exist_ok=True)

    # Split
    random.seed(42)
    image_ids = list(images.keys())
    random.shuffle(image_ids)
    split_idx = int(len(image_ids) * split_ratio)
    splits = {'train': image_ids[:split_idx], 'val': image_ids[split_idx:]}

    # Helper function
    def coco_to_yolo_bbox(bbox, w, h):
        x_c = (bbox[0] + bbox[2] / 2) / w
        y_c = (bbox[1] + bbox[3] / 2) / h
        w_n = bbox[2] / w
        h_n = bbox[3] / h
        return x_c, y_c, w_n, h_n

    count_processed = 0
    for split_name, img_ids in splits.items():
        for img_id in img_ids:
            img_info = images[img_id]
            filename = img_info['file_name']

            # COPY ẢNH: Dùng image_dir tìm được ở trên
            src = os.path.join(image_dir, filename)
            dst = os.path.join(output_path, 'images', split_name, filename)

            if os.path.exists(src):
                shutil.copy(src, dst)

                # CHỈ TẠO NHÃN KHI CÓ ẢNH
                label_name = os.path.splitext(filename)[0] + '.txt'
                label_path = os.path.join(output_path, 'labels', split_name, label_name)

                with open(label_path, 'w') as f:
                    if img_id in image_annotations:
                        for ann in image_annotations[img_id]:
                            # Zalo dataset thường category_id bắt đầu từ 1, YOLO cần từ 0
                            cat_id = ann['category_id'] - 1
                            cx, cy, w, h = coco_to_yolo_bbox(ann['bbox'], img_info['width'], img_info['height'])
                            # Giới hạn tọa độ trong khoảng [0, 1] đề phòng lỗi
                            cx, cy = max(0, min(1, cx)), max(0, min(1, cy))
                            w, h = max(0, min(1, w)), max(0, min(1, h))
                            f.write(f"{cat_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
                count_processed += 1
            else:
                # Trường hợp tên file trong json khác tên file thật (ít gặp nhưng có thể)
                pass

    print(f"✅ Đã xử lý xong {count_processed} ảnh.")

    # Create data.yaml
    data_yaml = {
        'path': output_path,
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(categories),
        'names': {c['id']-1: c['name'] for c in categories.values()}
    }

    yaml_path = os.path.join(output_path, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, sort_keys=False)

    print(f"📄 File cấu hình đã tạo tại: {yaml_path}")
    return yaml_path

# --- CHẠY CODE ---
# Path download từ kagglehub của bạn (không cần sửa)
downloaded_path = "/root/.cache/kagglehub/datasets/duongthanhduy/zaloaichallenge2020/versions/1"
yolo_dataset_path = "/content/yolo_dataset"

# Xóa folder cũ nếu có để tránh lỗi đè file
if os.path.exists(yolo_dataset_path):
    shutil.rmtree(yolo_dataset_path)

# Chạy hàm tự động
yaml_config_path = auto_convert_zalo_to_yolo(downloaded_path, yolo_dataset_path)

🔍 Đang quét dataset tại: /root/.cache/kagglehub/datasets/duongthanhduy/zaloaichallenge2020/versions/1
   ✓ Đã tìm thấy file nhãn: /root/.cache/kagglehub/datasets/duongthanhduy/zaloaichallenge2020/versions/1/zalo_clean_augmented_dataset/annotations.json
   ✓ Đã tìm thấy thư mục ảnh: /root/.cache/kagglehub/datasets/duongthanhduy/zaloaichallenge2020/versions/1/zalo_clean_augmented_dataset/images (8588 ảnh)

🚀 Bắt đầu chuyển đổi sang YOLO format...
✅ Đã xử lý xong 8588 ảnh.
📄 File cấu hình đã tạo tại: /content/yolo_dataset/data.yaml


## 3. Training

In [4]:
from ultralytics import YOLO

# Update path if needed, ensure it points to the generated data.yaml
print(f"Training with config: {yaml_config_path}")

# Path to the latest checkpoint (uncomment and update path if resuming)
# ckpt_path = "/content/last.pt"

# 2. Load the model
# NOTE: If loading from a checkpoint, change to: model = YOLO(ckpt_path)
# and make sure to set 'resume=True' in the training arguments below.
model = YOLO("yolo26m.pt")

# Train
results = model.train(
    data=yaml_config_path,
    epochs=100,
    imgsz=640,
    batch=16, # Caution: Use batch=8 or 4 if you get Out Of Memory (OOM) errors with Medium models
    plots=True,
    name="zalo_traffic_signs_v2",
    # resume=True # Uncomment this line to resume training
)

Training with config: /content/yolo_dataset/data.yaml
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=zalo_traffic_signs_v2, nbs=64, nms=False, opset=None, optim

KeyboardInterrupt: 

## 4. Download the weights to save your checkpoint. This allows you to resume training later if Google Colab's free tier disconnects before 100 epochs."

In [5]:
import os
from google.colab import files

# 1. Define the path
folder_path = "/content/runs/detect/zalo_traffic_signs_v2/weights"

# 2. Zip the folder
# -r: recursive (include all files)
# -j: junk paths (do not create the full folder structure inside the zip, just the files)
os.system(f"zip -r -j /content/zalo_weights.zip {folder_path}")

# 3. Trigger download
print("Downloading...")
files.download("/content/zalo_weights.zip")

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>